# AIS5281 Project Introduction - MachineLearning_for_DNAwalker
<em>Writer: Tse Kam Pui</em><br>
<em>Supervisor: Wang Zhisong / Co-Supervisor: Duane Loh</em><br>
<em>Last audited: July 30, 2026</em><br>
Code repository address: [MechineLearning_for_DNAwalker](https://github.com/GBGBGB780/MechineLearning_for_DNAwalker.git)

## 1. Introduction & Background

### 1.1 DNA Nanotechnology and the DNA Walker
DNA can act as a programmable structural material through sequence-specific base pairing.
This project studies a light-controlled DNA walker on a three-site track. Its dynamics are
represented by 14 accessible states whose transition rates depend on free-energy and
mechanical parameters under alternating visible and UV illumination.

### 1.2 Inverse Problem
The observed input is three synchronized fluorescence curves (FAM, TYE, and Cy5). The
task is to estimate seven latent parameters:

1. **$E_b$**: leg-track hybridization energy per base pair.
2. **$E_{b\_azo\_trans}$**: trans-azobenzene hairpin energy.
3. **$E_{b\_azo\_cis}$**: cis-azobenzene hairpin energy.
4. **$k_{mig}$**: leg migration rate.
5. **$k_0$**: zero-force dissociation rate.
6. **$drt\_z$**: unzipping force-coupling distance.
7. **$drt\_s$**: shearing force-coupling distance.

The inverse map is ill-posed: different parameter vectors can produce very similar
curves, and several parameters are weakly identifiable. A prediction must therefore be
interpreted as a curve-consistent candidate, not as uniquely recovered ground truth.

### 1.3 Approach and Final Scope
The repository compares a 1D-CNN and a patch-based Transformer as inverse predictors.
Each network supplies an initial candidate, after which model-agnostic Powell refinement
minimizes curve RMSE through the production `pysim` forward model.

The final local evidence includes a current-schema 10k five-seed comparison, a fixed 30k
nested 8k/16k/24k learning curve, checkpoint-independent identifiability and signal
diagnostics, and five-seed physics-refinement robustness on two experimental traces.
Neither controlled model-seed study establishes a preferred architecture. At 24k the
means are nearly tied, while refinement-start variability remains material on the
experimental generalization trace.

Synthetic recovery, warm-start, corrected speed, and capacity effects are outside the
final claim set; their source and historical outputs are absent from the streamlined tree.
The canonical-only suite contains 401 local tests. The public tree includes the selected
CNN/Transformer inference pairs and their required split manifest, but no training dataset.
Formal cross-platform certification still requires remote CI and fresh Linux/CUDA validation.

---
## 5. Methodology & Technical Implementation

Our workflow is divided into three rigorous phases: **Data Generation**, **Data Engineering**, and **AI Modeling, Training & Validation**.

## **Step1: Data Generation**
## Phase 1: Parameter Space Exploration via Latin Hypercube Sampling

### 1. Introduction

In the kinetic modeling of DNA nanorobots, accurate parameter estimation is crucial. We aim to determine 7 hidden physical parameters (e.g., Binding Energy $E_b$, Migration Rate $k_{mig}$) that govern the robot's motion.

To train our AI model effectively, we need a dataset that comprehensively covers the high-dimensional parameter space. Traditional random sampling often leads to clustering and gaps. Therefore, we adopt Latin Hypercube Sampling (LHS) to ensure maximizing stratification and coverage.

### 2. Parameter Definitions

We define the boundaries for the 7 parameters based on physical constraints and prior literature.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import qmc

from dnawalker.physics import simulator as pysim
from dnawalker.config import Config
from dnawalker.shared.parameters import load_npz_dataset
from dnawalker.data.preprocessing import normalize_per_sample

plt.style.use("seaborn-v0_8-whitegrid")

# Read the canonical parameter order and ranges from the production code.
config = Config()  # defaults to configs/common.ini via dnawalker.paths
param_names = list(pysim.PARAM_NAMES)
configured_ranges = {
    name.lower(): pair for name, pair in config.get_param_ranges().items()
}
bounds = np.asarray(
    [configured_ranges[name.lower()] for name in param_names],
    dtype=np.float64,
)

print(f"Total parameters to sample: {len(param_names)}")
for name, bound in zip(param_names, bounds):
    print(f"{name:>15}: [{bound[0]:.6g}, {bound[1]:.6g}]")

### 3. Latin Hypercube Sampling (LHS)

We use LHS to generate 10,000 samples.
Mathematically, for each dimension $j$, the range is divided into $N$ intervals of equal probability $1/N$. A sample is taken from each interval exactly once.

$$ X_j = \frac{\pi_j(0, 1, \dots, N-1) + U}{N} $$

where $\pi_j$ is a random permutation.

In [ ]:
num_samples = 10000

# Initialize LHS sampler
sampler = qmc.LatinHypercube(d=len(param_names), seed=42)

# Generate samples in [0, 1] range
sample_normalized = sampler.random(n=num_samples)

# Scale samples to physical bounds
# Formula: Y = min + (max - min) * sample_norm
lower_bounds = bounds[:, 0]
upper_bounds = bounds[:, 1]
Y_phys = qmc.scale(sample_normalized, lower_bounds, upper_bounds)

# Convert to DataFrame for visualization
df_params = pd.DataFrame(Y_phys, columns=param_names)
print(f"Generated shape: {df_params.shape}")
df_params.head()

### 4. Distribution Analysis

To verify the quality of our sampling, we visualize the distribution of the generated parameters. We expect a uniform (flat-top) distribution for each parameter, indicating equal probability coverage across the entire valid range.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(param_names):
    axes[i].hist(
        df_params[col],
        bins=50,
        color="skyblue",
        edgecolor="black",
        alpha=0.8,
    )
    axes[i].set_title(f"Distribution of {col}", fontsize=12)
    axes[i].set_ylabel("Count")
    axes[i].set_xlabel("Value")

for i in range(len(param_names), len(axes)):
    axes[i].axis("off")

plt.tight_layout()
plt.show()

The nearly equal marginal bin counts are the expected result of Latin Hypercube
stratification. They describe the **candidate sampling prior**, not the final retained
training distribution: simulation-validity checks, weak-signal filtering, and
activity-bin acceptance alter the final marginals.

## Phase 2: The Computational Microscope - Kinetic Simulation

### 1. Theoretical Framework: The Master Equation

The DNA nanorobot moves through a landscape of 14 intermediate states. Its dynamics are stochastic and governed by the Master Equation:

$$\frac{d\mathbf{P}(t)}{dt} = \mathbf{R} \cdot \mathbf{P}(t)$$

Where:

$\mathbf{P}(t)$ is the probability vector of the 14 states at time $t$.

$\mathbf{R}$ is the rate matrix (transition matrix), which depends on the free energy barriers between states.

### 2. Defining the Physics Engine

We implemented a simulator that translates the sampled physical parameters ($\theta$) into observable fluorescence signals. Below is the core logic for the Transition Rate Matrix construction and the Time Evolution loop.

In [ ]:
# Run the production forward simulator for one representative midpoint sample.
demo_params = bounds.mean(axis=1)
demo_signals, demo_dt = pysim.run_simulation(demo_params)

if demo_dt < 0 or not np.isfinite(demo_signals).all():
    raise RuntimeError("The representative parameter set produced an invalid simulation")

print(f"Parameter order: {param_names}")
print(f"Signal shape: {demo_signals.shape}; integration dt: {demo_dt:g} s")
print(
    "Per-channel signal ranges:",
    demo_signals.max(axis=1) - demo_signals.min(axis=1),
)

The cell above calls the canonical production implementation in [`dnawalker/core/pysim.py`](../../dnawalker/core/pysim.py), not a
pedagogical mock. The simulator constructs the complete transition network for 14
physical states, switches the visible/UV rate matrix according to the configured
10-minute phases, propagates the state probabilities, and maps them to the three
fluorescence channels. Physical constants, illumination durations, and simulation length
come from `configs/common.ini`.

## Phase 3: Data Synthesis & Experimental Observables

The 14-state probabilities are mapped to the observable FAM, TYE, and CY5 signals by the
production simulator. The canonical generator `dnawalker.data.generate` (run via
`python -m dnawalker.data.generate`) uses Latin Hypercube candidates, Python multiprocessing,
numerical-validity checks, a
weak-signal filter, and activity-based acceptance sampling.

The selected current artifact is stored as `artifacts/releases/retrain-3a5a494-ds557506e93079/training_dataset.npz` with:

- `X`: fluorescence curves in shape `(N, 3, 7801)`;
- `Y`: seven physical parameters in shape `(N, 7)`;
- `parameter_names`: explicit metadata used to validate and reorder label columns.

The next section inspects this real generated artifact.

## **Step 2: Data Engineering**

### 1. Validation and Filtering
`dnawalker.data.generate` rejects invalid simulations and weak-signal candidates before
saving. The
training loaders independently reject non-finite or extreme values so a malformed sample
cannot silently enter optimization.

### 2. Input Representation
The source artifact stores each sample as `(3, 7801)`. The CNN loader flattens this to
`23403` values for batching and reshapes it back to `(3, 7801)` before `Conv1d`; the
Transformer keeps the three-dimensional representation.

### 3. Input Normalization
Both architectures call the same `dnawalker.data.preprocessing.normalize_per_sample`
function. Each
sample is jointly z-score normalized across all channels and time points. No global
`StandardScaler` is fitted to the full dataset, so cross-split statistics are not leaked.

### 4. Target Transformation
`k0` is transformed with `log10`, then a `MinMaxScaler` fitted on the **training split
only** maps all seven targets to `[0.1, 0.9]`. The held-out validation and test labels are
transformed with that fitted scaler.

### Why the CNN Uses a Flattened Batch Representation

Flattening is an I/O representation, not a claim that time order is discarded. The CNN
receives a vector of length `3 x 7801` and immediately reshapes it to `(3, 7801)` before
temporal convolution. The Transformer loader retains `(3, 7801)` throughout.

### Why Invalid Values Are Rejected

Any non-finite input or label can make the loss and gradients non-finite. The dataset
generator and both loaders therefore validate samples before training. This is a data
integrity check, while the per-sample normalization described above is the actual model
preprocessing step.

The following inspection uses the selected 10,000-sample release training artifact. The code
still reads and reports the actual array shape instead of assuming a fixed sample count.
For UMAP, it deterministically samples first and downsamples the time axis before
normalization, avoiding the former full-dataset flattening and `StandardScaler` memory
spike.

In [ ]:
NPZ_FILENAME = Path("artifacts/releases/retrain-3a5a494-ds557506e93079/training_dataset.npz")
N_SAMPLES_FOR_VISUALIZATION = 1000
TIME_STRIDE = 10
VISUALIZATION_SEED = 42

In [ ]:
if not NPZ_FILENAME.exists():
    raise FileNotFoundError(
        f"{NPZ_FILENAME} not found. Obtain the reviewed legacy artifact described "
        "in docs/ARTIFACTS.md and verify artifacts.sha256. For a newly generated dataset, "
        "use a versioned output path and update NPZ_FILENAME explicitly."
    )

print(f"Loading {NPZ_FILENAME}...")
X_data, Y_data, param_names = load_npz_dataset(
    NPZ_FILENAME,
    param_names,
    allow_legacy_canonical=False,
)

expected_x_shape = (config.get_num_curves(), config.get_seq_length())
if X_data.shape[1:] != expected_x_shape:
    raise ValueError(
        f"Unexpected X sample shape {X_data.shape[1:]}; expected {expected_x_shape}"
    )

print("Loading successful")
print(f"Shape of X: {X_data.shape}")
print(f"Shape of Y: {Y_data.shape}")
print(f"Parameter order: {param_names}")

In [ ]:
# Validate in chunks so temporary boolean/absolute-value arrays stay small.
num_samples_original = X_data.shape[0]
safe_threshold = config.get_safe_threshold()
valid_mask = np.isfinite(Y_data).all(axis=1)
valid_mask &= (np.abs(Y_data) < safe_threshold).all(axis=1)

chunk_size = 256
for start in range(0, num_samples_original, chunk_size):
    stop = min(start + chunk_size, num_samples_original)
    block = X_data[start:stop]
    block_ok = np.isfinite(block).all(axis=(1, 2))
    block_ok &= (np.abs(block) < safe_threshold).all(axis=(1, 2))
    valid_mask[start:stop] &= block_ok

valid_indices = np.flatnonzero(valid_mask)
num_bad = num_samples_original - len(valid_indices)
if not len(valid_indices):
    raise ValueError("No valid samples remain after finite-value checks")

rng = np.random.default_rng(VISUALIZATION_SEED)
subset_size = min(N_SAMPLES_FOR_VISUALIZATION, len(valid_indices))
selected_indices = np.sort(
    rng.choice(valid_indices, size=subset_size, replace=False)
)

# Copy only the selected curves, then release the full ~1 GiB decompressed array.
X_selected = X_data[selected_indices].copy()
Y_subset = Y_data[selected_indices].copy()
del X_data, Y_data

print(f"Valid samples: {len(valid_indices)}; invalid samples: {num_bad}")
print(f"Deterministic visualization subset: {len(selected_indices)} samples")

In [ ]:
# Downsample only for exploratory UMAP, then use production per-sample normalization.
X_downsampled = X_selected[:, :, ::TIME_STRIDE]
X_normalized = normalize_per_sample(X_downsampled)
X_subset = X_normalized.reshape(len(X_normalized), -1)
del X_downsampled, X_normalized

print(f"UMAP feature shape after time stride {TIME_STRIDE}: {X_subset.shape}")

> **Visualization scope.** UMAP receives at most 1,000 deterministically selected samples,
> with the time axis downsampled by a factor of 10. It uses the same per-sample,
> joint-channel normalization as production. UMAP is exploratory only and does not affect
> training, validation, or reported model metrics.

In [ ]:
if Y_subset.ndim != 2 or Y_subset.shape[1] != len(param_names):
    raise ValueError(
        f"Unexpected label shape {Y_subset.shape}; "
        f"expected (N, {len(param_names)})"
    )

Y_vis = Y_subset
print(f"Label shape for visualization: {Y_vis.shape}")
pd.DataFrame(Y_vis, columns=param_names).describe().T

In [ ]:
try:
    import umap
except ImportError as exc:
    raise ImportError(
        "UMAP is optional. Install it with `python -m pip install umap-learn` "
        "to run this visualization cell."
    ) from exc

print("Starting UMAP...")
start_time = time.time()
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    n_components=2,
    random_state=VISUALIZATION_SEED,
    verbose=False,
)
embedding = reducer.fit_transform(X_subset)

print(f"UMAP completed in {time.time() - start_time:.2f} s")
print(f"Embedding shape: {embedding.shape}")

In [ ]:
print("Generating UMAP plots...")
fig, axes = plt.subplots(4, 2, figsize=(16, 24))
axes = axes.flatten()

for i, name in enumerate(param_names):
    scatter = axes[i].scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=Y_vis[:, i],
        cmap="Spectral",
        s=8,
        alpha=0.7,
    )
    axes[i].set_title(f"Color by: {name}", fontsize=12)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    plt.colorbar(scatter, ax=axes[i], label="Value")

for i in range(len(param_names), len(axes)):
    axes[i].axis("off")

plt.suptitle(f"UMAP of Simulation Data (N={len(embedding)})", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
SAMPLES_TO_PLOT = min(5, len(X_selected))
plot_rng = np.random.default_rng(VISUALIZATION_SEED)
plot_positions = np.sort(
    plot_rng.choice(len(X_selected), size=SAMPLES_TO_PLOT, replace=False)
)
time_axis = np.linspace(
    0.0,
    config.get_sim_duration_minutes(),
    X_selected.shape[2],
)

fig, axes = plt.subplots(
    SAMPLES_TO_PLOT,
    1,
    figsize=(11, 3 * SAMPLES_TO_PLOT),
    sharex=True,
)
axes = np.atleast_1d(axes)

for ax, position in zip(axes, plot_positions):
    curves = X_selected[position]
    params = Y_subset[position]
    original_index = selected_indices[position]

    ax.plot(time_axis, curves[0], label="FAM", color="green", linewidth=1.2)
    ax.plot(time_axis, curves[1], label="TYE", color="orange", linewidth=1.2)
    ax.plot(time_axis, curves[2], label="CY5", color="red", linewidth=1.2)
    values = dict(zip(param_names, params))
    ax.set_title(
        f"Sample {original_index}: "
        f"k_mig={values['k_mig']:.4f}, k0={values['k0']:.2e}"
    )
    ax.set_ylabel("Intensity")
    ax.grid(True, alpha=0.3)

axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Time (min)")
plt.tight_layout()
plt.show()

## **Step3: AI modeling, Training & Validation**

## Step 3 Implementation & Final Evidence

The repository implements two inverse models and a model-agnostic
physics-refinement stage. Legacy checkpoints remain compatibility-only, while
current-schema studies bind dataset, split, scaler, and checkpoint identity.

### 1. Two Model Architectures

| | **CNN (`dnawalker.cnn`)** | **Transformer (`dnawalker.transformer`)** |
|--|--|--|
| Architecture | 1D-Conv x4 -> Adaptive Pool -> FC x3 -> Sigmoid | PatchTST + Multi-Head Attention |
| Input | flattened $\mathbb{R}^{23403}$, reshaped before convolution | 3D $(3, 7801)$ |
| Trainable parameters | 4,381,319 (4.38M) | 3,243,271 (3.24M) |
| Raw FP32 weights | about 16.7 MiB | about 12.4 MiB |
| Analytical forward MAC/sample | about 38,067,168 | about 780,223,360 |
| Batch size / epoch cap | 256 / 2000 | 64 / 300 |
| Optimizer / Scheduler | Adam / ReduceLROnPlateau | AdamW / Cosine Warmup |

Both paths use fixed split membership, per-sample joint-channel normalization,
training-only target scaling, and a $\log_{10}$ transform on $k_0$.
Loading both initializers stores 7,624,590 trainable values, about 29.1 MiB
as raw FP32 weights before framework overhead.

The CNN has more stored parameters because its 16,384-to-256 dense projection
contains 4,194,304 weights. The Transformer has fewer weights but about 20.5x
the analytical forward MAC count because it applies temporal and cross-channel
attention to 78 patches. MAC excludes normalization, activation, pooling,
softmax, memory traffic, backward propagation, and device-specific kernels.

No comparable current-lineage wall-clock logs survive for training or raw
inference. CNN is structurally expected to be faster, but the project does not
claim a measured speed ratio. In the full application, repeated simulator calls
during multi-start Powell refinement usually dominate end-to-end latency.

A user workbook can be supplied without editing configuration:
`bash scripts/run_application.sh --exp /path/to/curves.xlsx --model transformer`.
The workbook contains time in minutes followed by FAM, TYE, and Cy5 columns.
`--model cnn` selects the lower-compute branch; `--model both` runs and verifies both.

### 2. Current-Schema 10k Five-Seed Result

| Architecture | Curve RMSE, mean +/- sample SD | Valid / invalid / extreme |
|--|--:|--:|
| CNN | 0.02124865 +/- 0.00108720 | 5000 / 0 / 0 |
| Transformer | 0.02118662 +/- 0.00064572 | 4989 / 8 / 3 |

The paired Transformer-minus-CNN mean is -0.00006203. An exploratory paired
test gives p=0.915 with a 95% interval crossing zero. This does not establish
superiority for either architecture.

### 3. Fixed 30k Nested Learning Curve

| Train | CNN mean +/- SD | Transformer mean +/- SD | Transformer - CNN (95% CI) | Decision |
|--:|--:|--:|--:|--|
| 8k | 0.01953252 +/- 0.00066847 | 0.02365253 +/- 0.00357211 | +0.00412001 [-0.00080583, +0.00904584] | Inconclusive |
| 16k | 0.01833638 +/- 0.00046472 | 0.02020585 +/- 0.00185304 | +0.00186948 [-0.00074115, +0.00448010] | Inconclusive |
| 24k | 0.01811960 +/- 0.00053594 | 0.01817831 +/- 0.00149517 | +0.00005871 [-0.00225976, +0.00237718] | Inconclusive |

Both architectures improve with more data. The 24k point estimate is a
near-tie, but five seeds do not place the full confidence interval inside the
predeclared +/-0.001 equivalence region.

Candidate rows originate from LHS, then pass physical-validity, weak-signal,
and activity-quota filters. The retained 30k rows are exactly activity-balanced
but not uniform over the seven-dimensional box. The strongest retained
correlation is `corr(E_b,E_b_azo_cis)=0.43853`.

Ten 8k numeric rows were restored from append-only session-log evidence. The
five original 8k Transformer checkpoint hashes and binaries are unavailable,
so they are not independently reproducible at artifact level.

### 4. Final Experimental-Fit Robustness

One checkpoint per architecture was selected only by minimum validation MSE
across model seeds 42-46: CNN 24k seed 43 and Transformer 24k seed 46. Both
were evaluated on the original and generalization traces with identical
settings and refinement RNG seeds 0-4.

| Architecture | Original median (range) | Generalization median (range) | Combined median (range) |
|--|--:|--:|--:|
| CNN | 0.016290 (0.007779-0.020323) | 0.016038 (0.015956-0.033571) | 0.016303 (0.016082-0.020675) |
| Transformer | 0.007806 (0.007796-0.008220) | 0.019721 (0.016087-0.024852) | 0.013767 (0.011946-0.016328) |

These values quantify optimizer-start sensitivity for one selected checkpoint
per architecture. They do not replace the model-seed comparison. The variation,
especially on the generalization trace, does not justify more model training.

![CNN three-channel experimental fit](../evidence/cnn_experimental_fit.png)

![Transformer three-channel experimental fit](../evidence/transformer_experimental_fit.png)

![Physics-refinement stability](../evidence/refinement_robustness.png)

### 5. Identifiability and Interpretation

At the documented reference point, the Fisher information matrix has condition
number approximately $1.3 \times 10^{15}$. Low curve RMSE is therefore evidence
of curve-space consistency, not unique microscopic parameter recovery.

Legacy evaluation remains a compatibility diagnostic. Repository restructuring
preserved legacy bytes and CPU/MPS behavior; differences among legacy, 10k, and
30k results arise from different dataset/checkpoint lineages, not from moving
source files.

## Capacity Ablation and Signal Diagnostics

The historical shrunk-CNN and enlarged-Transformer checkpoints lack current
dataset/scaler hashes and complete split/model provenance. Capacity effects are
therefore outside the final claim set, and their withdrawn outputs are not
retained in the final result hierarchy.

Checkpoint-independent diagnostics remain valid: about 95% of plotted signal
energy lies below 0.0023 Hz, one final pooled CNN feature has a local receptive
field of roughly 399 seconds, and median signal autocorrelation half-life is
about 1,303 seconds. These measurements describe the data; they do not select
an architecture.

<em>**Implementation status:** the canonical `dnawalker` package is the single
source of truth for pure-Python generation, training, prediction, refinement,
verification, and evaluation. The local scientific closeout includes the 10k
five-seed result, the 30k nested learning curve, and current-artifact
experimental-fit robustness. No additional model training is recommended for
the present scope. The selected inference bundle is included without the 30k
training dataset. Formal cross-platform certification still requires remote CI
and fresh Linux/CUDA validation. See `README.md`,
`docs/ARTIFACTS.md`, and `docs/TEST_CHECKLIST.md`.</em><br>
Last audited: July 30, 2026